# 🔷 4D Tesseract — Animated
### A hypercube rotating simultaneously in the XW and YZ planes

`4D Geometry` · `Dual Projection` · `Quasiperiodic Rotation` · `Higher Dimensions`

---

## What Is Actually Happening?

### Building Up from 1D to 4D

Each dimension is built by duplicating the previous shape and connecting all corresponding corners:

| Dimension | Name | Vertices | Edges | Faces | 3-Cells |
|-----------|------|----------|-------|-------|---------|
| 1D | Line segment | 2 | 1 | — | — |
| 2D | Square | 4 | 4 | 1 | — |
| 3D | Cube | 8 | 12 | 6 | 1 |
| **4D** | **Tesseract** | **16** | **32** | **24** | **8 full cubes** |

The tesseract has **eight cubic cells** — each face is a full 3D cube, just as each face of a cube is a full 2D square. All eight cubes are the same size in 4D space.

### How the Rotation Works

In 3D there are 3 independent rotation planes (XY, XZ, YZ). In 4D there are **six**: XY, XZ, XW, YZ, YW, ZW — where W is the fourth spatial axis. This animation applies two simultaneously:

```
R₁ : rotation in the XW plane at angular rate t      (mixes X and W coordinates)
R₂ : rotation in the YZ plane at angular rate 1.3t   (mixes Y and Z coordinates)
R  = R₂ · R₁   applied to all 16 vertices per frame
```

The two speeds (1.0 and 1.3) are chosen so the animation traces a dense set of orientations — never exactly repeating.

### The Double Projection

We collapse 4D → 2D in two stages:

1. **4D perspective divide on W** — points farther in W appear smaller (like 3D depth perspective), collapsing 4D → 3D. This produces the apparent "inner cube" (far in W) and "outer cube" (near in W).
2. **Orthographic drop of Z** — 3D → 2D for display.

### The "Inner Cube Popping Through" Mystery

This is not a rendering glitch. In 4D, those cubic cells are connected in a way that has no 3D analogue. As different cells rotate to the "front" of the W axis, their projected sizes rapidly exchange. In 4D space it is completely smooth and non-self-intersecting.

> The confusion you feel watching this is geometrically accurate. Your brain evolved to parse 3D scenes. Every time it locks on to a 3D interpretation, the rotation invalidates it — because this is not a 3D object. That feeling of oscillating confusion is exactly what a 4D rotation looks like from inside a 3D brain.

---

In [11]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import animation
from IPython.display import HTML, display
import warnings; warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':  '#03020a',
    'axes.facecolor':    '#03020a',
    'axes.edgecolor':    '#1a0a30',
    'text.color':        '#c8b8ff',
    'axes.labelcolor':   '#c8b8ff',
    'xtick.color':       '#5030a0',
    'ytick.color':       '#5030a0',
    'font.family':       'monospace',
})
print("Ready.")

Ready.


In [12]:
# 16 vertices of the tesseract: all (±1, ±1, ±1, ±1) combinations
verts4 = np.array([[x,y,z,w]
                   for x in (-1,1) for y in (-1,1)
                   for z in (-1,1) for w in (-1,1)], dtype=float)

# Edges: pairs of vertices differing in exactly ONE coordinate
edges = [(i,j) for i in range(16) for j in range(i+1,16)
         if np.sum(verts4[i] != verts4[j]) == 1]

print(f"Tesseract geometry: {len(verts4)} vertices, {len(edges)} edges  (expected: 16v, 32e)")

def rot4d(t):
    # Simultaneous rotation: XW-plane at rate t, YZ-plane at rate 1.3t
    c1, s1 = np.cos(t),       np.sin(t)
    c2, s2 = np.cos(t * 1.3), np.sin(t * 1.3)
    Rxw = np.array([[ c1, 0, 0, s1],
                    [  0, 1, 0,  0],
                    [  0, 0, 1,  0],
                    [-s1, 0, 0, c1]])
    Ryz = np.array([[1,   0,   0, 0],
                    [0,  c2,  s2, 0],
                    [0, -s2,  c2, 0],
                    [0,   0,   0, 1]])
    return Ryz @ Rxw

def project(v, dist=3.5):
    # Stage 1: 4D perspective divide on W axis
    pts = v.copy()
    w   = pts[:, 3]
    s   = 1.0 / (dist - w)          # points far in W appear smaller
    p3  = pts[:, :3] * s[:, None]   # collapse 4D → 3D
    # Stage 2: orthographic drop of Z (3D → 2D)
    return p3[:, :2], p3[:, 2]

FRAMES = 90
fig, ax = plt.subplots(figsize=(9, 9), facecolor='#03020a')
ax.set_facecolor('#03020a')
ax.set_aspect('equal')
ax.axis('off')
ax.set_xlim(-2.2, 2.2);  ax.set_ylim(-2.2, 2.2)
ax.set_title("4D TESSERACT  ·  Simultaneous XW × YZ rotation\n"
             "Edge brightness = depth after 4D→3D projection",
             color='#3a86ff', fontsize=11, pad=10)

lo = [ax.plot([], [], lw=1.2, alpha=0.0)[0] for _ in edges]
do = [ax.plot([], [], 'o', ms=4,  alpha=0.0)[0] for _ in range(16)]

def init():
    [l.set_data([], []) for l in lo]
    [d.set_data([], []) for d in do]
    return lo + do

def upd(f):
    t   = f / FRAMES * 2 * np.pi
    R   = rot4d(t)
    pts = (R @ verts4.T).T
    xy, zv = project(pts)
    dep = (zv - zv.min()) / (zv.max() - zv.min() + 1e-9)
    for obj, (i, j) in zip(lo, edges):
        da = (dep[i] + dep[j]) / 2
        obj.set_data([xy[i,0], xy[j,0]], [xy[i,1], xy[j,1]])
        obj.set_color((da*0.3, da*0.6, 1.0 - da*0.7))
        obj.set_alpha(0.35 + 0.65*da)
        obj.set_linewidth(0.7 + 1.5*da)
    for obj, k in zip(do, range(16)):
        obj.set_data([xy[k,0]], [xy[k,1]])
        obj.set_color((0.5 + 0.5*dep[k], 0.5*dep[k], 1.0))
        obj.set_alpha(0.6 + 0.4*dep[k])
        obj.set_markersize(3 + 4*dep[k])
    return lo + do

anim = animation.FuncAnimation(fig, upd, frames=FRAMES,
                                init_func=init, interval=50, blit=True)
plt.close()
display(HTML(anim.to_jshtml(fps=20, default_mode='loop')))

print("The inner cube popping through the outer is not a glitch.")
print("In 4D, those cells connect in a way no 3D analogy can describe.")

Tesseract geometry: 16 vertices, 32 edges  (expected: 16v, 32e)


The inner cube popping through the outer is not a glitch.
In 4D, those cells connect in a way no 3D analogy can describe.


---

## 💡 Three Profound Truths

**01 —** Simple rules produce infinite complexity. Chaos is not the absence of order — it is order we cannot compress.

**02 —** The universe has at least 4 dimensions. Every object you have ever seen is a shadow of something higher.

**03 —** Primes are not random. They obey a law no human has proved. The Riemann Hypothesis still waits.

---
*▸ end of descent ◂*